In [1]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm import tqdm

# ==========================================
# 1. Instance-Conditioned DistArc Loss (PROPOSAL A)
# ==========================================
class ProposalADistArcLoss(nn.Module):
    def __init__(self, in_features, out_features, m=0.4, lam=0.005, r_min=5.0, r_max=20.0, alignment_weight=0.1): 
        super(ProposalADistArcLoss, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.m = m
        self.lam = lam 
        
        self.r_min = r_min
        self.r_max = r_max
        self.alignment_weight = alignment_weight 
        
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        
        self.W = nn.Parameter(torch.FloatTensor(in_features, out_features))
        nn.init.kaiming_uniform_(self.W)

    def forward(self, x, radii, labels):
        batch_size = x.size(0)
        
        x_norm = F.normalize(x, p=2, dim=1)
        W_norm = F.normalize(self.W, p=2, dim=0)
        
        cosine = torch.matmul(x_norm, W_norm).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        target_logit = cosine[torch.arange(batch_size), labels]
        
        sine = torch.sqrt((1.0 - torch.pow(target_logit, 2)).clamp(min=1e-7))
        marginal_cosine = target_logit * self.cos_m - sine * self.sin_m 

        # --- PROPOSAL A: UNCERTAINTY ALIGNMENT ---
        uncertainty = (1.0 - target_logit).detach() 
        u_norm = (uncertainty - uncertainty.min()) / (uncertainty.max() - uncertainty.min() + 1e-6)
        ideal_radii = self.r_min + (self.r_max - self.r_min) * u_norm
        alignment_loss = F.mse_loss(radii.squeeze(), ideal_radii)
        
        # --- DISTARC RADIAL/ANGULAR MATH ---
        radii_sq = torch.pow(radii, 2) 
        dist_sq = 2.0 * radii_sq * (1.0 - cosine + 1e-6) 
        
        num_exp = marginal_cosine - (self.lam * dist_sq[torch.arange(batch_size), labels])
        denom_terms = cosine - (self.lam * dist_sq)
        denom_terms[torch.arange(batch_size), labels] = marginal_cosine - (self.lam * dist_sq[torch.arange(batch_size), labels])
        
        base_loss = -num_exp + torch.logsumexp(denom_terms, dim=1)
        return base_loss.mean() + (self.alignment_weight * alignment_loss)

    def predict(self, x):
        x_norm = F.normalize(x, p=2, dim=1)
        W_norm = F.normalize(self.W, p=2, dim=0)
        cosine = torch.matmul(x_norm, W_norm)
        return torch.argmax(cosine, dim=1)

# ==========================================
# 2. iResNet50 Architecture Components
# ==========================================
def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=dilation, groups=groups, bias=False, dilation=dilation)

def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

class IBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1, base_width=64, dilation=1):
        super(IBasicBlock, self).__init__()
        self.bn1 = nn.BatchNorm2d(inplanes, eps=1e-05)
        self.conv1 = conv3x3(inplanes, planes)
        self.bn2 = nn.BatchNorm2d(planes, eps=1e-05)
        self.prelu = nn.PReLU(planes)
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn3 = nn.BatchNorm2d(planes, eps=1e-05)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x
        out = self.bn1(x)
        out = self.conv1(out)
        out = self.bn2(out)
        out = self.prelu(out)
        out = self.conv2(out)
        out = self.bn3(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return out

class IResNet(nn.Module):
    fc_scale = 7 * 7 # Expects 112x112 input
    def __init__(self, block, layers, dropout=0, num_features=512, zero_init_residual=False, groups=1, width_per_group=64):
        super(IResNet, self).__init__()
        self.inplanes = 64
        self.dilation = 1
        self.groups = groups
        self.base_width = width_per_group
        
        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(self.inplanes, eps=1e-05)
        self.prelu = nn.PReLU(self.inplanes)
        self.layer1 = self._make_layer(block, 64, layers[0], stride=2)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.bn2 = nn.BatchNorm2d(512 * block.expansion, eps=1e-05)
        self.dropout = nn.Dropout(p=dropout, inplace=True)
        self.fc = nn.Linear(512 * block.expansion * self.fc_scale, num_features)
        self.features = nn.BatchNorm1d(num_features, eps=1e-05)
        nn.init.constant_(self.features.weight, 1.0)
        self.features.weight.requires_grad = False

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm2d(planes * block.expansion, eps=1e-05),
            )
        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample, self.groups, self.base_width, self.dilation))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups=self.groups, base_width=self.base_width, dilation=self.dilation))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.prelu(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.bn2(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        x = self.features(x)
        return x

def iresnet50(**kwargs):
    return IResNet(IBasicBlock, [3, 4, 14, 3], **kwargs)

# ==========================================
# 3. Model Wrapper (Proposal A Implementation)
# ==========================================
class iResNet50_ProposalA(nn.Module):
    def __init__(self, embedding_size=512, r_min=5.0, r_max=20.0):
        super(iResNet50_ProposalA, self).__init__()
        
        self.backbone = iresnet50(num_features=512)
        self.projection = nn.Linear(512, embedding_size)
        
        self.r_min = r_min
        self.r_max = r_max
        self.radial_head = nn.Sequential(
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
            nn.Sigmoid() 
        )
            
    def forward(self, x):
        features = self.backbone(x) 
        
        embeddings = self.projection(features)
        radii_pred = self.r_min + (self.r_max - self.r_min) * self.radial_head(features)
        
        return embeddings, radii_pred

# ==========================================
# 4. Data Loading 
# ==========================================
def get_dataloaders(batch_size):
    train_transform = transforms.Compose([
        transforms.Resize((112, 112)), 
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((112, 112)), 
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, transform=train_transform, download=True)
    val_dataset = torchvision.datasets.CIFAR100(root='./data', train=False, transform=val_transform, download=True)
    
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                                               num_workers=2, pin_memory=True, persistent_workers=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                                             num_workers=2, pin_memory=True, persistent_workers=True)
    return train_loader, val_loader

# ==========================================
# 5. Main Training Loop (AMP Enabled + Fixed Resume)
# ==========================================
def train_model(
    embedding_size=512, 
    epochs=150, 
    batch_size=128, 
    lr=1e-2,           
    weight_decay=5e-4, 
    lam=0.005,
    save_dir='./checkpoints',
    resume_path=None
):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_classes = 100 
    
    os.makedirs(save_dir, exist_ok=True)
    
    best_save_path = os.path.join(save_dir, f'best_iresnet50_proposalA_dim{embedding_size}.pth')
    latest_save_path = os.path.join(save_dir, f'latest_iresnet50_proposalA_dim{embedding_size}.pth')

    print(f"--- Starting AMP Training: PROPOSAL A ---")
    print(f"Model: iResNet50 | Emb Size: {embedding_size} | Target: CIFAR-100")

    model = iResNet50_ProposalA(embedding_size=embedding_size).to(device)
    train_loader, val_loader = get_dataloaders(batch_size)

    if torch.cuda.device_count() > 1:
        print(f"Awesome, utilizing {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
    
    criterion = ProposalADistArcLoss(in_features=embedding_size, out_features=num_classes, lam=lam, m=0.4).to(device)
    
    optimizer = optim.SGD([
        {'params': model.parameters()},
        {'params': criterion.parameters()}
    ], lr=lr, weight_decay=weight_decay, momentum=0.9)

    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=75, gamma=0.1)
    scaler = torch.amp.GradScaler('cuda')
    
    best_acc = 0.0
    start_epoch = 0
    
    # --- FIXED RESUME LOGIC ---
    target_load_path = resume_path if resume_path else latest_save_path
    
    if target_load_path and os.path.exists(target_load_path):
        print(f"Found existing checkpoint! Loading from {target_load_path}...")
        checkpoint = torch.load(target_load_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        criterion.load_state_dict(checkpoint['criterion_state_dict'])
        start_epoch = checkpoint['epoch']
        best_acc = checkpoint.get('best_acc', 0.0)
        print(f"Resuming training from Epoch {start_epoch + 1} with Best Acc so far: {best_acc:.2f}%")
    elif resume_path:
        print(f"WARNING: You provided a resume_path, but no file was found at {resume_path}!")
        print("Starting training from Epoch 1 instead.")

    for epoch in range(start_epoch, epochs):
        model.train()
        criterion.train()
        
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False, mininterval=2.0)
        
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda'):
                embeddings, radii_pred = model(images)
                
            loss = criterion(embeddings.float(), radii_pred.float(), labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            
            params_to_clip = list(model.parameters()) + list(criterion.parameters())
            torch.nn.utils.clip_grad_norm_(params_to_clip, max_norm=5.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            pbar.set_postfix({'loss': f"{running_loss/(pbar.n+1):.4f}"})
            
        model.eval()
        criterion.eval()
        
        correct = 0
        total = 0
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False, mininterval=2.0)
            for images, labels in val_pbar:
                images, labels = images.to(device), labels.to(device)
                
                with torch.amp.autocast('cuda'):
                    embeddings, _ = model(images)
                
                predicted = criterion.predict(embeddings.float())
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
        acc = 100 * correct / total
        scheduler.step()
        
        print(f"Epoch {epoch+1}/{epochs} Complete | Train Loss: {running_loss/len(train_loader):.4f} | Val Acc: {acc:.2f}% | Next LR: {scheduler.get_last_lr()[0]}")
        
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'criterion_state_dict': criterion.state_dict(),
            'best_acc': best_acc,
            'embedding_size': embedding_size
        }
            
        torch.save(checkpoint, latest_save_path)
        
        if acc > best_acc:
            best_acc = acc
            print(f"--> New Best Accuracy! ({best_acc:.2f}%) Saving BEST model...")
            torch.save(checkpoint, best_save_path)

# ==========================================
# 6. CIFAR-100 Proposal A Visualizer
# ==========================================
def verify_proposal_a_topography_cifar100(target_class=19, checkpoint_path='./checkpoints/latest_iresnet50_proposalA_dim512.pth'):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    model = iResNet50_ProposalA(embedding_size=512).to(device)
    
    if not os.path.exists(checkpoint_path):
        print(f"Error: Checkpoint not found at {checkpoint_path}")
        return
        
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = {k.replace('module.', ''): v for k, v in checkpoint['model_state_dict'].items()}
    model.load_state_dict(state_dict)
    model.eval()

    _, val_loader = get_dataloaders(batch_size=1024)
    images, labels = next(iter(val_loader))
    
    class_names = val_loader.dataset.classes
    class_name = class_names[target_class]
    
    mask = (labels == target_class)
    images = images[mask].to(device)
    labels = labels[mask].to(device)

    if len(images) < 10:
        print(f"Not enough images found in this batch for class {class_name}. Visualizer skipped.")
        return

    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            embeddings, radii = model(images)

    radii = radii.squeeze().cpu().float().numpy()
    images_np = images.cpu().numpy()

    sorted_indices = np.argsort(radii)
    simple_indices = sorted_indices[:5]   
    complex_indices = sorted_indices[-5:] 

    def unnormalize(img_arr):
        mean = np.array([0.5, 0.5, 0.5])
        std = np.array([0.5, 0.5, 0.5])
        img_arr = img_arr.transpose(1, 2, 0)
        img_arr = std * img_arr + mean
        img_arr = np.clip(img_arr, 0, 1)
        return img_arr

    fig, axes = plt.subplots(2, 5, figsize=(15, 7))
    fig.suptitle(f"Proposal A Verification: Class '{class_name}' (CIFAR-100)", fontsize=20, fontweight='bold')

    for i, idx in enumerate(simple_indices):
        ax = axes[0, i]
        img = unnormalize(images_np[idx])
        ax.imshow(img)
        ax.set_title(f"Rad: {radii[idx]:.2f}", color='green', fontsize=14, fontweight='bold')
        ax.axis('off')
        if i == 0:
            ax.text(-5, 56, "PROPOSAL A:\nSIMPLE (Origin)", fontsize=14, color='green', ha='right', va='center', rotation=90, fontweight='bold')

    for i, idx in enumerate(complex_indices):
        ax = axes[1, i]
        img = unnormalize(images_np[idx])
        ax.imshow(img)
        ax.set_title(f"Rad: {radii[idx]:.2f}", color='red', fontsize=14, fontweight='bold')
        ax.axis('off')
        if i == 0:
            ax.text(-5, 56, "PROPOSAL A:\nCOMPLEX (Shell)", fontsize=14, color='red', ha='right', va='center', rotation=90, fontweight='bold')

    plt.tight_layout()
    plt.subplots_adjust(left=0.15, top=0.85)
    plt.show()

if __name__ == '__main__':
    # ---------------------------------------------------------
    # The path to your downloaded Kaggle dataset weights
    # ---------------------------------------------------------
    my_downloaded_model_path = '/kaggle/input/models/pradneshkalkar/latest-iresnet50-proposala-dim512/pytorch/latest_iresnet50_proposala_dim512/1/latest_iresnet50_proposalA_dim512.pth' 
    
    # 1. Resume Training
    train_model(
        embedding_size=512, 
        epochs=150, 
        batch_size=128, 
        lr=1e-2,
        resume_path=my_downloaded_model_path
    )
    


--- Starting AMP Training: PROPOSAL A ---
Model: iResNet50 | Emb Size: 512 | Target: CIFAR-100


100%|██████████| 169M/169M [00:08<00:00, 20.6MB/s]


Awesome, utilizing 2 GPUs!
Found existing checkpoint! Loading from /kaggle/input/models/pradneshkalkar/latest-iresnet50-proposala-dim512/pytorch/latest_iresnet50_proposala_dim512/1/latest_iresnet50_proposalA_dim512.pth...
Resuming training from Epoch 58 with Best Acc so far: 51.31%


Epoch 58/150 Complete | Train Loss: 3.8158 | Val Acc: 42.29% | Next LR: 0.01


Epoch 59/150 Complete | Train Loss: 3.8024 | Val Acc: 42.29% | Next LR: 0.01


...

Epoch 110/150 Complete | Train Loss: 3.5916 | Val Acc: 40.24% | Next LR: 0.001


Epoch 111/150 Complete | Train Loss: 3.5910 | Val Acc: 40.60% | Next LR: 0.001


Epoch 112/150 Complete | Train Loss: 3.5878 | Val Acc: 41.02% | Next LR: 0.001


Epoch 113/150 Complete | Train Loss: 3.5824 | Val Acc: 41.62% | Next LR: 0.001


Epoch 114/150 [Train]:  88%|████████▊ | 344/391 [04:31<09:21, 11.95s/it, loss=3.6184]